# DiffPhysDrone: Multi-Agent Simulation with Obstacles

This notebook demonstrates vision-based agile flight training using differentiable physics for multi-agent drone simulations with virtual obstacles.

**Paper**: Learning Vision-based Agile Flight via Differentiable Physics (Nature Machine Intelligence 2025)

**Project**: [DiffPhysDrone Web](https://henryhuyu.github.io/DiffPhysDrone_Web/)

## Features:
- Multi-agent swarm simulations
- Dynamic obstacle generation and avoidance
- Vision-based control with depth perception
- Differentiable physics simulation
- CUDA-accelerated training


## 1. Environment Setup and Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install matplotlib tqdm tensorboard
!pip install ninja  # For faster compilation

# Check CUDA availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA not available. Training will be very slow.")

In [ ]:
# Clone the repository
!git clone https://github.com/2302660/DiffPhysDrone.git
%cd DiffPhysDrone

In [ ]:
# Build CUDA extensions
!pip install -e src/

## 2. Import Modules and Setup

In [ ]:
import math
import random
import numpy as np
from collections import defaultdict
from matplotlib import pyplot as plt
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import argparse

# Import custom modules
from env_cuda import Env
from model import Model

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Configuration and Hyperparameters

In [ ]:
# Configuration class to replace argparse for Colab
class Config:
    def __init__(self):
        # Training parameters
        self.batch_size = 64  # Reduced for Colab memory constraints
        self.num_iters = 5000  # Reduced for demo
        self.lr = 1e-3
        self.grad_decay = 0.4
        self.timesteps = 150
        
        # Loss coefficients
        self.coef_v = 1.0
        self.coef_speed = 0.0
        self.coef_v_pred = 2.0
        self.coef_collide = 5.0  # Increased for multi-agent
        self.coef_obj_avoidance = 2.0  # Increased for multi-agent
        self.coef_d_acc = 0.01
        self.coef_d_jerk = 0.001
        self.coef_d_snap = 0.0
        self.coef_ground_affinity = 0.0
        self.coef_bias = 0.0
        
        # Environment parameters
        self.speed_mtp = 1.0
        self.fov_x_half_tan = 0.82  # Multi-agent config
        self.cam_angle = 10
        
        # Multi-agent specific settings
        self.single = False  # Enable multi-agent
        self.gate = True     # Enable gate obstacles
        self.ground_voxels = False
        self.scaffold = False
        self.random_rotation = False
        self.yaw_drift = False
        self.no_odom = False
        
        # Demo/visualization
        self.resume = None

args = Config()
print("Configuration loaded for multi-agent simulation with obstacles")

## 4. Environment and Model Initialization

In [ ]:
# Initialize environment
env = Env(
    batch_size=args.batch_size, 
    width=64, 
    height=48, 
    grad_decay=args.grad_decay, 
    device=device,
    fov_x_half_tan=args.fov_x_half_tan, 
    single=args.single,
    gate=args.gate, 
    ground_voxels=args.ground_voxels,
    scaffold=args.scaffold, 
    speed_mtp=args.speed_mtp,
    random_rotation=args.random_rotation, 
    cam_angle=args.cam_angle
)

# Initialize model
if args.no_odom:
    model = Model(7, 6)  # Without odometry
else:
    model = Model(7+3, 6)  # With odometry
    
model = model.to(device)

# Initialize optimizer and scheduler
optim = AdamW(model.parameters(), args.lr)
sched = CosineAnnealingLR(optim, args.num_iters, args.lr * 0.01)

print(f"Environment initialized with {args.batch_size} parallel simulations")
print(f"Multi-agent mode: {not args.single}")
print(f"Gate obstacles: {args.gate}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

## 5. Visualization Functions

In [ ]:
def visualize_environment_state(env, sample_idx=0):
    """Visualize the current state of the environment"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Render depth image
    depth, _ = env.render(1/15)
    depth_vis = depth[sample_idx].cpu().numpy()
    depth_vis = np.clip(depth_vis / 10.0, 0, 1)  # Normalize for visualization
    
    axes[0, 0].imshow(depth_vis, cmap='viridis')
    axes[0, 0].set_title('Depth Image (Agent View)')
    axes[0, 0].axis('off')
    
    # Plot 3D trajectory
    ax_3d = fig.add_subplot(2, 2, 2, projection='3d')
    
    # Current positions
    pos = env.p[sample_idx:sample_idx+env.n_drones_per_group].cpu().numpy()
    target_pos = env.p_target[sample_idx:sample_idx+env.n_drones_per_group].cpu().numpy()
    
    ax_3d.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c='blue', s=100, label='Current Position')
    ax_3d.scatter(target_pos[:, 0], target_pos[:, 1], target_pos[:, 2], 
                  c='red', s=100, marker='x', label='Target Position')
    
    # Draw obstacles (simplified)
    balls = env.balls[sample_idx].cpu().numpy()
    for i, ball in enumerate(balls[:10]):  # Show first 10 obstacles
        if ball[3] > 0:  # Valid obstacle
            ax_3d.scatter(ball[0], ball[1], ball[2], c='gray', s=ball[3]*1000, alpha=0.3)
    
    ax_3d.set_xlabel('X')
    ax_3d.set_ylabel('Y')
    ax_3d.set_zlabel('Z')
    ax_3d.legend()
    ax_3d.set_title('3D Environment (Top View)')
    
    # Velocity plot
    velocities = env.v[sample_idx:sample_idx+env.n_drones_per_group].cpu().numpy()
    axes[1, 0].bar(range(len(velocities)), [np.linalg.norm(v) for v in velocities])
    axes[1, 0].set_title('Agent Velocities (Magnitude)')
    axes[1, 0].set_xlabel('Agent ID')
    axes[1, 0].set_ylabel('Speed (m/s)')
    
    # Distance to obstacles
    vec_to_nearest = env.find_vec_to_nearest_pt()[sample_idx:sample_idx+env.n_drones_per_group]
    distances = torch.norm(vec_to_nearest, dim=1).cpu().numpy()
    axes[1, 1].bar(range(len(distances)), distances)
    axes[1, 1].set_title('Distance to Nearest Obstacle')
    axes[1, 1].set_xlabel('Agent ID')
    axes[1, 1].set_ylabel('Distance (m)')
    axes[1, 1].axhline(y=env.margin[sample_idx].cpu().item(), color='r', linestyle='--', label='Safety Margin')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

def plot_training_progress(losses_dict, success_rates):
    """Plot training progress"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Total loss
    axes[0, 0].plot(losses_dict['total'])
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    
    # Success rate
    axes[0, 1].plot(success_rates)
    axes[0, 1].set_title('Success Rate (Collision-free)')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Success Rate')
    axes[0, 1].grid(True)
    axes[0, 1].set_ylim(0, 1)
    
    # Component losses
    for key in ['velocity', 'collision', 'obj_avoidance']:
        if key in losses_dict:
            axes[1, 0].plot(losses_dict[key], label=key)
    axes[1, 0].set_title('Component Losses')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Speed statistics
    if 'avg_speed' in losses_dict and 'max_speed' in losses_dict:
        axes[1, 1].plot(losses_dict['avg_speed'], label='Average Speed')
        axes[1, 1].plot(losses_dict['max_speed'], label='Max Speed')
        axes[1, 1].set_title('Speed Statistics')
        axes[1, 1].set_xlabel('Iteration')
        axes[1, 1].set_ylabel('Speed (m/s)')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()

print("Visualization functions defined")

## 6. Quick Environment Demo

In [ ]:
# Reset environment and visualize initial state
env.reset()
print(f"Environment reset with {env.n_drones_per_group} agents per group")
print(f"Batch size: {args.batch_size}")
print(f"Total agents: {args.batch_size}")
print(f"Max speed: {env.max_speed[0].item():.2f} m/s")
print(f"Drone radius: {env.drone_radius:.3f} m")

# Visualize initial environment state
visualize_environment_state(env, sample_idx=0)

## 7. Training Loop with Multi-Agent Obstacle Avoidance

In [ ]:
# Helper functions for training
def barrier(x: torch.Tensor, v_to_pt):
    """Barrier function for obstacle avoidance"""
    return (v_to_pt * (1 - x).relu().pow(2)).mean()

def is_save_iter(i, save_freq=250):
    """Determine if this is a save iteration"""
    if i < 1000:
        return (i + 1) % save_freq == 0
    return (i + 1) % 500 == 0

def smooth_dict(ori_dict, scaler_q):
    """Add values to smoothing queue"""
    for k, v in ori_dict.items():
        scaler_q[k].append(float(v))

print("Helper functions defined")

In [ ]:
# Training loop
print("Starting multi-agent training with obstacle avoidance...")

# Training tracking
scaler_q = defaultdict(list)
losses_history = defaultdict(list)
success_history = []

# Control timestep
ctl_dt = 1 / 15

pbar = tqdm(range(args.num_iters), desc="Training")

for i in pbar:
    # Reset environment and model state
    env.reset()
    model.reset()
    
    # History tracking
    p_history = []
    v_history = []
    target_v_history = []
    vec_to_pt_history = []
    v_preds = []
    vid = []
    h = None
    
    # Action buffer for control delay
    act_lag = 1
    act_buffer = [env.act] * (act_lag + 1)
    target_v_raw = env.p_target - env.p
    
    # Simulation timesteps
    for t in range(args.timesteps):
        # Render depth and optical flow
        depth, flow = env.render(ctl_dt)
        p_history.append(env.p)
        vec_to_pt_history.append(env.find_vec_to_nearest_pt())
        
        # Save video frames for visualization
        if is_save_iter(i) and t % 10 == 0:  # Sample frames
            vid.append(depth[0])
        
        # Update target velocity
        target_v_raw = env.p_target - env.p.detach()
        env.run(act_buffer[t], ctl_dt, target_v_raw)
        
        # Prepare state for neural network
        R = env.R
        fwd = env.R[:, :, 0].clone()
        up = torch.zeros_like(fwd)
        fwd[:, 2] = 0
        up[:, 2] = 1
        fwd = F.normalize(fwd, 2, -1)
        R = torch.stack([fwd, torch.cross(up, fwd), up], -1)
        
        # Calculate target velocity with speed limits
        target_v_norm = torch.norm(target_v_raw, 2, -1, keepdim=True)
        target_v_unit = target_v_raw / target_v_norm
        target_v = target_v_unit * torch.minimum(target_v_norm, env.max_speed)
        
        # Prepare state vector
        state = [
            torch.squeeze(target_v[:, None] @ R, 1),
            env.R[:, 2],
            env.margin[:, None]
        ]
        local_v = torch.squeeze(env.v[:, None] @ R, 1)
        if not args.no_odom:
            state.insert(0, local_v)
        state = torch.cat(state, -1)
        
        # Normalize depth input with noise
        x = 3 / depth.clamp_(0.3, 24) - 0.6 + torch.randn_like(depth) * 0.02
        x = F.max_pool2d(x[:, None], 4, 4)
        
        # Neural network forward pass
        act, values, h = model(x, state, h)
        
        # Process actions
        a_pred, v_pred, *_ = (R @ act.reshape(args.batch_size, 3, -1)).unbind(-1)
        v_preds.append(v_pred)
        act = (a_pred - v_pred - env.g_std) * env.thr_est_error[:, None] + env.g_std
        act_buffer.append(act)
        
        # Store history
        v_history.append(env.v)
        target_v_history.append(target_v)
    
    # Compute losses
    p_history = torch.stack(p_history)
    v_history = torch.stack(v_history)
    target_v_history = torch.stack(target_v_history)
    vec_to_pt_history = torch.stack(vec_to_pt_history)
    act_buffer = torch.stack(act_buffer)
    v_preds = torch.stack(v_preds)
    
    # Velocity tracking loss
    v_history_cum = v_history.cumsum(0)
    v_history_avg = (v_history_cum[30:] - v_history_cum[:-30]) / 30
    delta_v = torch.norm(v_history_avg - target_v_history[1:1-30], 2, -1)
    loss_v = F.smooth_l1_loss(delta_v, torch.zeros_like(delta_v))
    
    # Velocity prediction loss
    loss_v_pred = F.mse_loss(v_preds, v_history.detach())
    
    # Control regularization
    jerk_history = act_buffer.diff(1, 0).mul(15)
    loss_d_acc = act_buffer.pow(2).sum(-1).mean()
    loss_d_jerk = jerk_history.pow(2).sum(-1).mean()
    
    # Obstacle avoidance losses
    distance = torch.norm(vec_to_pt_history, 2, -1) - env.margin
    with torch.no_grad():
        v_to_pt = (-torch.diff(distance, 1, 1) * 135).clamp_min(1)
    loss_obj_avoidance = barrier(distance[:, 1:], v_to_pt)
    loss_collide = F.softplus(distance[:, 1:].mul(-32)).mul(v_to_pt).mean()
    
    # Ground avoidance
    loss_ground_affinity = p_history[..., 2].relu().pow(2).mean()
    
    # Combined loss
    loss = (args.coef_v * loss_v + 
            args.coef_obj_avoidance * loss_obj_avoidance +
            args.coef_d_acc * loss_d_acc +
            args.coef_d_jerk * loss_d_jerk +
            args.coef_v_pred * loss_v_pred +
            args.coef_collide * loss_collide +
            args.coef_ground_affinity * loss_ground_affinity)
    
    # Check for NaN
    if torch.isnan(loss):
        print("Loss is NaN, stopping training...")
        break
    
    # Optimization step
    optim.zero_grad()
    loss.backward()
    optim.step()
    sched.step()
    
    # Track metrics
    with torch.no_grad():
        speed_history = v_history.norm(2, -1)
        success = torch.all(distance.flatten(0, 1) > 0, 0)
        success_rate = success.sum() / args.batch_size
        
        # Store metrics
        metrics = {
            'total': loss,
            'velocity': loss_v,
            'collision': loss_collide,
            'obj_avoidance': loss_obj_avoidance,
            'success_rate': success_rate,
            'avg_speed': speed_history.mean(),
            'max_speed': speed_history.max()
        }
        
        smooth_dict(metrics, scaler_q)
        success_history.append(float(success_rate))
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss:.3f}',
            'success': f'{success_rate:.2f}',
            'avg_speed': f'{speed_history.mean():.2f}'
        })
        
        # Store history every 25 iterations
        if (i + 1) % 25 == 0:
            for k, v in scaler_q.items():
                if v:  # Check if list is not empty
                    losses_history[k].append(sum(v) / len(v))
            scaler_q.clear()
    
    # Visualization every 500 iterations
    if (i + 1) % 500 == 0:
        print(f"\nIteration {i+1}: Loss = {loss:.4f}, Success Rate = {success_rate:.3f}")
        visualize_environment_state(env, sample_idx=0)
        
        if len(losses_history['total']) > 1:
            plot_training_progress(losses_history, success_history[::25])

print("\nTraining completed!")

## 8. Conclusion and Next Steps

This notebook demonstrates a complete multi-agent drone simulation with obstacle avoidance using differentiable physics. The key features include:

### What was accomplished:
- ✅ Multi-agent drone swarm simulation
- ✅ Dynamic obstacle generation and avoidance
- ✅ Vision-based control using depth perception
- ✅ Differentiable physics for end-to-end training
- ✅ Real-time visualization and monitoring
- ✅ Customizable scenarios and parameters

### Key Components:
1. **Environment**: Generates complex 3D environments with various obstacle types
2. **Vision System**: Renders depth images from drone perspective
3. **Neural Network**: Processes visual input and state to generate control actions
4. **Physics Simulation**: CUDA-accelerated differentiable dynamics
5. **Multi-agent Coordination**: Handles swarm behaviors and inter-agent avoidance

### For further development:
- Increase training iterations for better performance
- Experiment with different network architectures
- Add more complex obstacle patterns
- Implement formation flying behaviors
- Test with real drone hardware

### Citation:
```
@article{zhang2025learning,
  title={Learning vision-based agile flight via differentiable physics},
  author={Zhang, Yuang and Hu, Yu and Song, Yunlong and Zou, Danping and Lin, Weiyao},
  journal={Nature Machine Intelligence},
  pages={1--13},
  year={2025},
  publisher={Nature Publishing Group}
}
```